<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Способы оплаты</h2>

----

### Вариант задания 16



<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Описание задачи: 
Создать базовый класс PaymentMethod в C#, который будет представлять 
различные способы оплаты. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов должны быть реализованы новые атрибуты и методы, а также 
переопределены некоторые методы базового класса для демонстрации 
полиморфизма. 
Требования к базовому классу PaymentMethod: 

• Атрибуты: 
1. ID способа оплаты (PaymentMethodId)
2. Название способа оплаты (MethodName)
3. Минимальная сумма (MinAmount). 

• Методы: 
1. ProcessPayment(decimal amount): метод для обработки платежа 
указанной суммы. 
2. CheckMinimumAmount(decimal amount): метод для проверки 
минимальной суммы платежа. 
3. GetPaymentDetails(): метод для получения деталей способа оплаты.
 
Требования к производным классам: 
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные 
атрибуты, 
такие 
как 
URL платежной системы (PaymentUrl). 
Метод ProcessPayment() должен быть переопределен для включения URL 
платежной системы в процесс оплаты. 
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные 
атрибуты, 
такие 
как 
Банковские 
данные 
(BankData). 
Метод CheckMinimumAmount() должен быть переопределен для проверки 
минимальной суммы платежа с учетом банковских комиссий. 
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать 
дополнительные атрибуты, такие как Место выдачи наличных 
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен 
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [18]:
using System;
using System.Collections.Generic;

public interface IRefundable
{
    bool Refund(decimal amount);
}

public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    protected string _currency;
    protected bool _isEnabled;
    protected int _transactionCount;
    protected string _providerName;
    protected DateTime _lastTransactionDate;

    public string ProviderName { get { return _providerName; } set { _providerName = value; } }
    public DateTime LastTransactionDate { get { return _lastTransactionDate; } set { _lastTransactionDate = value; } }
    public int PaymentMethodId { get { return _paymentMethodId; } set { _paymentMethodId = value; } }
    public string MethodName { get { return _methodName; } set { _methodName = value; } }
    public decimal MinAmount { get { return _minAmount; } set { if (value >= 0) _minAmount = value; } }
    public string Currency { get { return _currency; } set { _currency = value; } }
    public bool IsEnabled { get { return _isEnabled; } set { _isEnabled = value; } }

    public PaymentMethod(int id, string name, decimal minAmount, string currency, string provider)
    {
        this.PaymentMethodId = id;
        this.MethodName = name; 
        this.MinAmount = minAmount;
        this.Currency = currency;
        this.ProviderName = provider;
        this.IsEnabled = true;
        this._transactionCount = 0;
        this._lastTransactionDate = DateTime.MinValue;
    }

    public virtual void ProcessPayment(decimal amount)
    {
        if (!IsEnabled) { Console.WriteLine($"Метод '{MethodName}' отключен."); return; }
        if (CheckMinimumAmount(amount))
        {
            _transactionCount++;
            _lastTransactionDate = DateTime.Now;
            Console.WriteLine($"[УСПЕХ] Платеж через {MethodName} ({ProviderName}) на {amount} {Currency} обработан.");
        }
        else
        {
            Console.WriteLine($"[ОТКАЗ] Платеж на {amount} {Currency} отклонен. Мин. сумма: {MinAmount} {Currency}.");
        }
    }

    public virtual void ProcessPayment(decimal amount, string promoCode)
    {
        Console.WriteLine($"[ИНФО] Получен запрос на оплату {amount} {Currency} с промокодом '{promoCode}'.");
        Console.WriteLine($"[ИНФО] Промокод распознан. Применяется скидка 10%.");
        decimal discountedAmount = amount * 0.9m;
        Console.WriteLine($"[ИНФО] Итоговая сумма к оплате после скидки: {discountedAmount} {Currency}.");
        ProcessPayment(discountedAmount);
    }

    public virtual bool CheckMinimumAmount(decimal amount) { return amount >= MinAmount; }

    public virtual void ShowLastTransactionInfo()
    {
        if (_lastTransactionDate == DateTime.MinValue) Console.WriteLine("Транзакций еще не было.");
        else Console.WriteLine($"Последняя операция по {MethodName} была: {_lastTransactionDate}");
    }

    public int GetTransactionCount() { return _transactionCount; }
    public void Disable() { IsEnabled = false; }
}

public class OnlinePayment : PaymentMethod, IRefundable
{
    private string _paymentUrl;
    private decimal _transactionFee;
    private int _maxDailyTransactions;
    private bool _is3DSecureEnabled;

    public string PaymentUrl { get { return _paymentUrl; } set { _paymentUrl = value; } }
    public int MaxDailyTransactions { get { return _maxDailyTransactions; } set { _maxDailyTransactions = value; } }
    public bool Is3DSecureEnabled { get { return _is3DSecureEnabled; } set { _is3DSecureEnabled = value; } }

    public OnlinePayment(int id, string name, decimal minAmount, string currency, string provider, string url, decimal fee, int maxTrans) 
        : base(id, name, minAmount, currency, provider)
    {
        this.PaymentUrl = url;
        this._transactionFee = fee;
        this.MaxDailyTransactions = maxTrans;
        this._is3DSecureEnabled = true;
    }

    public override void ProcessPayment(decimal amount)
    {
        if (GetTransactionCount() >= MaxDailyTransactions)
        {
            Console.WriteLine($"[ОТКАЗ] Превышен дневной лимит операций ({MaxDailyTransactions}) для {MethodName}.");
            return;
        }
    
        if (!IsEnabled)
        {
            Console.WriteLine($"[ОТКАЗ] Метод '{MethodName}' отключен. Оплата невозможна.");
            return;
        }
        
        base.ProcessPayment(amount);
        Console.WriteLine($"--> Деньги списаны по URL: {PaymentUrl} (Комиссия сервиса: {_transactionFee} {Currency})");
    }

    public override void ProcessPayment(decimal amount, string promoCode)
    {
        Console.WriteLine($"[ОНЛАЙН] Проверка промокода {promoCode} через API... Успешно!");
        base.ProcessPayment(amount, promoCode);
    }

    public void Toggle3DSecure()
    {
        _is3DSecureEnabled = !_is3DSecureEnabled;
        Console.WriteLine($"Защита 3D Secure для {MethodName} теперь: {(_is3DSecureEnabled ? "ВКЛ" : "ВЫКЛ")}");
    }

    public bool Refund(decimal amount)
    {
        Console.WriteLine($"[ВОЗВРАТ] {amount} {Currency} будут возвращены на карту в течение 3 дней.");
        return true;
    }
}

public class BankTransfer : PaymentMethod
{
    private decimal _bankData;
    private string _swiftCode;
    private string _correspondentAccount;
    private int _processingTimeHours;

    public decimal BankData { get { return _bankData; } set { _bankData = value; } }
    public string SwiftCode { get { return _swiftCode; } set { _swiftCode = value; } }
    public string CorrespondentAccount { get { return _correspondentAccount; } set { _correspondentAccount = value; } }

    public BankTransfer(int id, string name, decimal minAmount, string currency, string provider, decimal bankData, string swift, string corrAcc, int time) 
        : base(id, name, minAmount, currency, provider)
    {
        this.BankData = bankData;
        this._swiftCode = swift;
        this._correspondentAccount = corrAcc;
        this._processingTimeHours = time;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount / 100 * BankData;
        decimal amountAfterCommission = amount - commission;
        if (amountAfterCommission >= MinAmount) return true;
        Console.WriteLine($"[ОТКАЗ] Недостаточно средств. Комиссия: {commission}. Остаток: {amountAfterCommission} (нужно {MinAmount})");
        return false;
    }

    public override void ShowLastTransactionInfo()
    {
        base.ShowLastTransactionInfo();
        Console.WriteLine($"(Детали банка: SWIFT {_swiftCode}, Корр.счет {_correspondentAccount})");
    }

    public void ValidateBankDetails()
    {
        if (_swiftCode.Length >= 8) Console.WriteLine($"[ПРОВЕРКА] SWIFT-код {_swiftCode} валиден.");
        else Console.WriteLine("[ОШИБКА] Неверный формат SWIFT-кода.");
    }

    public void EstimateArrivalTime()
    {
        Console.WriteLine($"Ожидаемое время зачисления средствами банка: {_processingTimeHours} ч.");
    }
}

public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;
    private string _operatorName;
    private string _cashierName;
    private decimal _cashInDrawer;

    public string CashPickupPoint { get { return _cashPickupPoint; } set { _cashPickupPoint = value; } }
    public string CashierName { get { return _cashierName; } set { _cashierName = value; } }
    public decimal CashInDrawer { get { return _cashInDrawer; } set { _cashInDrawer = value; } }

    public CashPayment(int id, string name, decimal minAmount, string currency, string provider, string pickupPoint, string opName, string cashier, decimal cash) 
        : base(id, name, minAmount, currency, provider)
    {
        this._cashPickupPoint = pickupPoint;
        this._operatorName = opName;
        this._cashierName = cashier;
        this._cashInDrawer = cash;
    }

    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount); 
        _cashInDrawer += amount;
        Console.WriteLine($"--> Деньги приняты. Теперь у кассира {_cashierName}: {_cashInDrawer} {Currency}");
    }

    public void RequestCashier()
    {
        Console.WriteLine($"[ВЫЗОВ] Кассир {_cashierName} приглашается к окну.");
    }

    public void PrintReceipt()
    {
        Console.WriteLine($"Чек распечатан оператором {_operatorName}");
    }
}

public class CryptoPayment : OnlinePayment
{
    private double _networkFee;
    private int _confirmationsRequired;

    public double NetworkFee { get { return _networkFee; } set { _networkFee = value; } }
    public int ConfirmationsRequired { get { return _confirmationsRequired; } set { _confirmationsRequired = value; } }

    public CryptoPayment(int id, string name, decimal minAmount, string currency, string provider, string url, decimal fee, int maxTrans, double netFee, int conf) 
        : base(id, name, minAmount, currency, provider, url, fee, maxTrans)
    {
        this._networkFee = netFee;
        this._confirmationsRequired = conf;
    }

    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"[КРИПТО] Попытка оплаты {amount} {Currency}");
        Console.WriteLine($"[КРИПТО] Добавляем комиссию сети: {_networkFee} {Currency}");
        decimal totalAmount = amount + (decimal)_networkFee;
        Console.WriteLine($"[КРИПТО] Итоговая сумма к списанию: {totalAmount} {Currency}");
        Console.WriteLine($"[КРИПТО] Ожидание {_confirmationsRequired} подтверждений сети...");
        base.ProcessPayment(totalAmount); 
    }

    public void CheckNetworkCongestion()
    {
        Console.WriteLine($"[СТАТУС СЕТИ] Загруженность средняя. Комиссия сети: {_networkFee} {Currency}.");
    }

    public void VerifyBlockchain()
    {
        Console.WriteLine($"Проверка транзакции в блокчейне... Подтверждено.");
    }
}

public class PaymentTerminal<T> where T : PaymentMethod
{
    private List<T> _availableMethods = new List<T>();
    public void AddMethod(T method) { _availableMethods.Add(method); }
    public void ProcessAllAvailable(decimal amount)
    {
        Console.WriteLine($"\n--- ТЕРМИНАЛ: попытка оплаты {amount} через все методы ---");
        foreach (var method in _availableMethods) { method.ProcessPayment(amount); }
    }
}

public class Customer
{
    public string Name { get; set; }
    private decimal _balance;
    private string _loyaltyTier;

    public decimal Balance { get { return _balance; } set { _balance = value; } }
    public string LoyaltyTier { get { return _loyaltyTier; } set { _loyaltyTier = value; } }

    public Customer(string name, decimal balance, string tier)
    {
        Name = name;
        _balance = balance;
        _loyaltyTier = tier;
    }

    public void MakePurchase(PaymentMethod method, decimal amount)
    {
        Console.WriteLine($"\n[КЛИЕНТ] {Name} (статус {_loyaltyTier}) пытается заплатить {amount} руб.");
        method.ProcessPayment(amount);
    }

    public void RequestRefund(IRefundable refundableMethod, decimal amount)
    {
        Console.WriteLine($"\n[КЛИЕНТ] {Name} запрашивает возврат средств.");
        refundableMethod.Refund(amount);
    }

    public void AddFunds(decimal amount)
    {
        _balance += amount;
        Console.WriteLine($"[БАЛАНС] Счет {Name} пополнен на {amount}. Текущий баланс: {_balance} руб.");
    }
}


BankTransfer bank = new BankTransfer(2, "Банковский перевод", 50, "RUB", "Сбербанк", 5, "SABRRUMM", "40702810...", 24);
CashPayment cash = new CashPayment(3, "Наличные", 100, "RUB", "Газпромбанк", "Касса №1", "Ольга", "Лариса", 5000);
OnlinePayment online = new OnlinePayment(1, "Онлайн оплата", 10, "RUB", "ЮKassa", "https://pay.ru", 2, 10);
CryptoPayment crypto = new CryptoPayment(4, "Крипто-оплата", 500, "USDT", "Binance", "https://crypto.gate", 1, 5, 0.5, 3);

Customer buyer = new Customer("Кирилл", 10000, "Gold");

Console.WriteLine("=== Демонстрация работы банковского перевода и наличных ===");
bank.EstimateArrivalTime();
buyer.MakePurchase(bank, 200);
cash.PrintReceipt();
buyer.MakePurchase(cash, 150);

Console.WriteLine("\n=== Применение промокода к онлайн-оплате ===");
online.ProcessPayment(1000, "SALE2024");

Console.WriteLine("\n=== Возврат средств через онлайн-оплату ===");
buyer.RequestRefund(online, 50);

Console.WriteLine("\n=== Работа с крипто-оплатой ===");
crypto.VerifyBlockchain();
buyer.MakePurchase(crypto, 1000);

Console.WriteLine("\n=== Использование универсального терминала ===");
PaymentTerminal<PaymentMethod> terminal = new PaymentTerminal<PaymentMethod>();
terminal.AddMethod(bank);
terminal.AddMethod(cash);
terminal.AddMethod(crypto);
terminal.AddMethod(online);
terminal.ProcessAllAvailable(600);

Console.WriteLine("\n=== Статистика транзакций ===");
Console.WriteLine($"Всего успешных транзакций через Онлайн: {online.GetTransactionCount()}");
Console.WriteLine($"Всего успешных транзакций через Крипто: {crypto.GetTransactionCount()}");
Console.WriteLine($"Всего успешных транзакций через Банк: {bank.GetTransactionCount()}");
Console.WriteLine($"Всего успешных транзакций через Наличные: {cash.GetTransactionCount()}");

Console.WriteLine("\n=== Отключение метода оплаты ===");
bank.ShowLastTransactionInfo();
online.Disable();
buyer.MakePurchase(online, 100);

=== Демонстрация работы банковского перевода и наличных ===
Ожидаемое время зачисления средствами банка: 24 ч.

[КЛИЕНТ] Кирилл (статус Gold) пытается заплатить 200 руб.
[УСПЕХ] Платеж через Банковский перевод (Сбербанк) на 200 RUB обработан.
Чек распечатан оператором Ольга

[КЛИЕНТ] Кирилл (статус Gold) пытается заплатить 150 руб.
[УСПЕХ] Платеж через Наличные (Газпромбанк) на 150 RUB обработан.
--> Деньги приняты. Теперь у кассира Лариса: 5150 RUB

=== Применение промокода к онлайн-оплате ===
[ОНЛАЙН] Проверка промокода SALE2024 через API... Успешно!
[ИНФО] Получен запрос на оплату 1000 RUB с промокодом 'SALE2024'.
[ИНФО] Промокод распознан. Применяется скидка 10%.
[ИНФО] Итоговая сумма к оплате после скидки: 900.0 RUB.
[УСПЕХ] Платеж через Онлайн оплата (ЮKassa) на 900.0 RUB обработан.
--> Деньги списаны по URL: https://pay.ru (Комиссия сервиса: 2 RUB)

=== Возврат средств через онлайн-оплату ===

[КЛИЕНТ] Кирилл запрашивает возврат средств.
[ВОЗВРАТ] 50 RUB будут возвращены на карт